# Case 02 · LoRA from scratch

**Goal:** implement LoRA yourself in ~30 lines, prove it matches full fine-tuning at a fraction of the
trainable parameters, and understand *why* it works. This is the engine behind `peft` and QLoRA (Case 03).

Pair with `animation.html`. Runs on CPU.

---
### The idea in one line
> Freeze the pretrained weight **W**. Learn a low-rank update **ΔW = B·A·(α/r)** and use **W + ΔW**.
> Only the skinny matrices **A** (r×in) and **B** (out×r) are trained.

Why it's allowed: empirically, the *change* a task needs is low-rank — it lives in a few directions,
so two thin matrices can represent it.

In [ ]:
import torch, torch.nn as nn, math, copy
import matplotlib.pyplot as plt
torch.manual_seed(0)

## 1 · The LoRA layer (the whole trick)

Wrap a frozen `nn.Linear` with a trainable low-rank side-path. Note **B is initialized to zero** so the
adapted model *starts identical* to the pretrained one — a safe, no-surprise start.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad_(False)   # FREEZE W
        in_f, out_f = base.in_features, base.out_features
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)          # small random
        self.B = nn.Parameter(torch.zeros(out_f, r))                # zero -> ΔW=0 at start
    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale

## 2 · Helpers + pretrain on Task A
Same toy setup as Case 01 so you can compare directly.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(), nn.Linear(64,1))
def task_data(phase, n=256):
    x = torch.linspace(-math.pi, math.pi, n).unsqueeze(1); return x, torch.sin(x+phase)
def train(model, x, y, steps, lr):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    lossf = nn.MSELoss(); hist=[]
    for _ in range(steps):
        opt.zero_grad(); l=lossf(model(x),y); l.backward(); opt.step(); hist.append(l.item())
    return hist
def trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

xa,ya = task_data(0.0); model = make_model(); train(model, xa, ya, 1500, 1e-2)
print('pretrained. total params:', sum(p.numel() for p in model.parameters()))

## 3 · Inject LoRA and compare to full fine-tuning
New Task B arrives. Approach 1: full fine-tune (train everything). Approach 2: LoRA (train only adapters).

In [ ]:
def inject_lora(model, r=4, alpha=8):
    return nn.Sequential(*[LoRALinear(m,r,alpha) if isinstance(m,nn.Linear) else m for m in model])

xb,yb = task_data(1.2)
full = copy.deepcopy(model);              h_full = train(full, xb, yb, 400, 1e-3)
lora = inject_lora(copy.deepcopy(model)); h_lora = train(lora, xb, yb, 400, 1e-2)

print(f'FULL  fine-tune: {trainable(full):5d} trainable params -> loss {h_full[-1]:.5f}')
print(f'LoRA  fine-tune: {trainable(lora):5d} trainable params -> loss {h_lora[-1]:.5f}')
print(f'LoRA trains {100*trainable(lora)/trainable(full):.1f}% of the params')

In [ ]:
plt.figure(figsize=(6,3))
plt.plot(h_full, label=f'full  ({trainable(full)} params)')
plt.plot(h_lora, label=f'LoRA  ({trainable(lora)} params)')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss (log)'); plt.legend()
plt.title('LoRA matches full fine-tuning with far fewer trainable params'); plt.show()

## 4 · Rank sweep — how small can r go?
Lower r = cheaper = fewer params. Find the rank where quality stops improving — that's your sweet spot.

In [ ]:
ranks = [1,2,4,8,16]; finals=[]; params=[]
for r in ranks:
    m = inject_lora(copy.deepcopy(model), r=r, alpha=2*r)
    h = train(m, xb, yb, 400, 1e-2); finals.append(h[-1]); params.append(trainable(m))
fig,ax=plt.subplots(1,2,figsize=(9,3))
ax[0].plot(ranks, finals, 'o-'); ax[0].set_xlabel('rank r'); ax[0].set_ylabel('final loss'); ax[0].set_title('quality vs rank')
ax[1].plot(ranks, params, 's-', color='green'); ax[1].set_xlabel('rank r'); ax[1].set_ylabel('trainable params'); ax[1].set_title('cost vs rank')
plt.tight_layout(); plt.show()
for r,f,p in zip(ranks,finals,params): print(f'r={r:2d}  loss={f:.5f}  params={p}')

## 5 · The base is untouched — swap adapters per task
Because W is frozen, you can keep **one small adapter per task** and load whichever you need.
This is the first real weapon against catastrophic forgetting (Case 06): the shared knowledge in W is
never overwritten.

In [ ]:
print('Base weight unchanged by LoRA training?',
      torch.equal(lora[0].base.weight, model[0].weight))

# adapter is tiny -> cheap to store many of them
adapter_state = {k:v for k,v in lora.state_dict().items() if ('A' in k or 'B' in k)}
print('adapter tensors you would save per task:', list(adapter_state.keys()))

## 6 · Merge for free inference
At deploy time you can fold the adapter back into W so there's **zero** extra inference cost.

In [ ]:
# W_merged = W + (B @ A) * scale  -> a normal Linear again
lin0 = lora[0]
with torch.no_grad():
    merged_W = lin0.base.weight + (lin0.B @ lin0.A) * lin0.scale
print('merged weight shape:', tuple(merged_W.shape), '-> identical forward, no adapter needed at inference')

## 7 · Takeaways → Case 03
1. LoRA = freeze W, learn low-rank **B·A**. Tiny trainable footprint, near-equal quality.
2. **B=0 init** → safe start. **Merge** → free inference.
3. Frozen base + per-task adapters → a path around forgetting (Case 06).

Next: **Case 03** swaps our toy `LoRALinear` for the real `peft` library and applies **QLoRA**
(LoRA on a 4-bit quantized model) to an actual Llama — the practical workflow you'll reuse for the capstone.

➡️ Do `challenge.md`, then `cases/03_qlora_real_llm/` (built next).